# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q huggingface_hub

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [4]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [5]:
df = pd.read_parquet(parquet_path)

print(df.shape)

(9841378, 30)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
import pandas as pd
import numpy as np

# Copy the original data
feature_df = df.copy()

# -----------------------------
# Engineered Features
# -----------------------------

# Click-Through Rate (CTR)
feature_df["ctr"] = np.where(
    feature_df["gsc_impressions"] > 0,
    feature_df["gsc_clicks"] / feature_df["gsc_impressions"],
    0
)

# Engagement Rate
feature_df["engagement_rate"] = np.where(
    feature_df["ga4_sessions"] > 0,
    feature_df["ga4_engaged_sessions"] / feature_df["ga4_sessions"],
    0
)

# Average engagement time per session
feature_df["avg_engagement_sec"] = np.where(
    feature_df["ga4_sessions"] > 0,
    feature_df["ga4_total_engagement_sec"] / feature_df["ga4_sessions"],
    0
)

# -----------------------------
# Categorical Handling
# -----------------------------

feature_df["position_bucket"] = pd.cut(
    feature_df["gsc_avg_position"],
    bins=[0, 10, 20, 50, float("inf")],
    labels=["Top10", "11-20", "21-50", "50+"]
)

# Encode the categorical feature
feature_df["position_bucket"] = (
    feature_df["position_bucket"]
    .cat.codes
)

# -----------------------------
# Missing Values
# -----------------------------

numeric_cols = [
    "ctr",
    "engagement_rate",
    "avg_engagement_sec"
]

feature_df[numeric_cols] = feature_df[numeric_cols].fillna(0)

# -----------------------------
# Final Feature Vector
# -----------------------------

feature_vector = feature_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "engagement_rate",
        "avg_engagement_sec",
        "position_bucket"
    ]
]

feature_vector.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,engagement_rate,avg_engagement_sec,position_bucket
0,20,0,3.350000,0.000,0.0,0.0,0
1,1,0,0.000000,0.000,0.0,0.0,-1
2,125,1,4.928000,0.008,0.0,0.0,0
3,7,0,4.000000,0.000,0.0,0.0,0
4,11,0,2.272727,0.000,0.0,0.0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [10]:
import pandas as pd

summary = pd.DataFrame({
    "Feature": feature_vector.columns,
    "Data Type": feature_vector.dtypes.astype(str).values,
    "Missing Count": feature_vector.isnull().sum().values,
})

summary["Missing %"] = (
    summary["Missing Count"] / len(feature_vector) * 100
).round(2)

summary["Unique Values"] = [
    feature_vector[col].nunique()
    for col in feature_vector.columns
]

summary["Mean"] = [
    feature_vector[col].mean()
    if pd.api.types.is_numeric_dtype(feature_vector[col])
    else None
    for col in feature_vector.columns
]

summary["Category"] = [
    "Categorical"
    if pd.api.types.is_object_dtype(feature_vector[col])
    or pd.api.types.is_categorical_dtype(feature_vector[col])
    else "Numerical"
    for col in feature_vector.columns
]

summary


/tmp/ipykernel_4482/598505892.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  or pd.api.types.is_categorical_dtype(feature_vector[col])


,Feature,Data Type,Missing Count,Missing %,Unique Values,Mean,Category
0,gsc_impressions,int64,0,0.00,4854,28.518119,Numerical
1,gsc_clicks,int64,0,0.00,140,0.083508,Numerical
2,gsc_avg_position,float64,6230317,63.31,457684,15.826651,Numerical
3,ctr,float64,0,0.00,15529,0.001130,Numerical
4,engagement_rate,float64,0,0.00,275,0.001464,Numerical
5,avg_engagement_sec,float64,0,0.00,5824,0.229394,Numerical
6,position_bucket,int8,0,0.00,5,-0.384165,Numerical


In [11]:
# Handle missing values

for col in feature_vector.columns:

    if feature_vector[col].isnull().sum() == 0:
        continue

    # Numerical features
    if pd.api.types.is_numeric_dtype(feature_vector[col]):
        feature_vector[col] = feature_vector[col].fillna(
            feature_vector[col].median()
        )

    # Categorical features
    else:
        feature_vector[col] = feature_vector[col].fillna(
            feature_vector[col].mode()[0]
        )

print(feature_vector.isnull().sum())

gsc_impressions       0
gsc_clicks            0
gsc_avg_position      0
ctr                   0
engagement_rate       0
avg_engagement_sec    0
position_bucket       0
dtype: int64


/tmp/ipykernel_4482/3705904951.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feature_vector[col] = feature_vector[col].fillna(


In [12]:
summary_after = pd.DataFrame({
    "Feature": feature_vector.columns,
    "Missing After Fill": feature_vector.isnull().sum().values
})

summary_after

,Feature,Missing After Fill
0,gsc_impressions,0
1,gsc_clicks,0
2,gsc_avg_position,0
3,ctr,0
4,engagement_rate,0
5,avg_engagement_sec,0
6,position_bucket,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [13]:
# Keywords that may indicate leakage
leakage_keywords = [
    "label",
    "target",
    "future",
    "next",
    "product",
    "flag"
]

# Find matching columns
leakage_columns = []

for col in df.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in leakage_keywords):
        leakage_columns.append(col)

print("Potential leakage columns:")
print(leakage_columns)

Potential leakage columns:
[]


In [14]:
# Features actually used by the model
selected_features = feature_vector.columns.tolist()

print("Selected Features:")
print(selected_features)

# Any leakage features accidentally selected?
used_leakage = [
    col for col in selected_features
    if col in leakage_columns
]

if len(used_leakage) == 0:
    print("\nLeakage Test Passed")
    print("No label-derived, future-window, or product flag columns are used.")
else:
    print("\nLeakage Detected!")
    print(used_leakage)

Selected Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'engagement_rate', 'avg_engagement_sec', 'position_bucket']

Leakage Test Passed
No label-derived, future-window, or product flag columns are used.


In [15]:
historical_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]

unexpected = [
    col for col in selected_features
    if col not in historical_features
    and col not in [
        "ctr",
        "engagement_rate",
        "avg_engagement_sec",
        "position_bucket"
    ]
]

print("Unexpected features:")
print(unexpected if unexpected else "None")

Unexpected features:
None


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [16]:
# Features selected for the model
selected_features = feature_vector.columns.tolist()

# Columns excluded from modeling
excluded = [col for col in df.columns if col not in selected_features]

# Build reasons automatically
reasons = []

for col in excluded:
    if col.endswith("_hash_id"):
        reasons.append("Identifier only; excluded from modeling.")
    elif col == "report_date":
        reasons.append("Used only for filtering the time window.")
    elif col.startswith("ai_") or col == "sessions_ai":
        reasons.append("AI referral metric; outside the chosen lane.")
    else:
        reasons.append("Not selected for the baseline feature vector.")

excluded_features = pd.DataFrame({
    "Excluded Field": excluded,
    "Reason": reasons
})

excluded_features


,Excluded Field,Reason
0,report_date,Used only for filtering the time window.
1,client_hash_id,Identifier only; excluded from modeling.
2,content_hash_id,Identifier only; excluded from modeling.
3,client_has_gsc,Not selected for the baseline feature vector.
4,client_has_ga4,Not selected for the baseline feature vector.
5,gsc_data_available,Not selected for the baseline feature vector.
6,ga4_data_available,Not selected for the baseline feature vector.
7,gsc_sum_position,Not selected for the baseline feature vector.
8,ga4_pageviews,Not selected for the baseline feature vector.
9,ga4_sessions,Not selected for the baseline feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.